In [2]:
import pandas as pd

from src import PROJECT_DIR, logging
from src import utils as src_utils

from discovery_utils.getters import crunchbase
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts,
    google,
    google_slides,
    
)

import re

PROJECT_NAME = src_utils.PROJECT_NAME
OUTPUT_DIR = src_utils.OUTPUT_DIR / "mission_radar"


# Mission Radar charts

## Crunchbase investments

In [33]:
from discovery_utils.utils.io import remap_dict
investment_stages = {'early_stage': ['pre_seed',
  'seed',
  'angel',
  'series_a',
  'series_b',
  'convertible_note',
  'equity_crowdfunding',
  'product_crowdfunding',
  'non_equity_assistance',
  'initial_coin_offering'],
 'growth_stage': ['series_c',
  'series_d',
  'series_e',
  'series_f',
  'series_g',
  'series_h',
  'series_i',
  'series_j'],
 'late_stage': ['private_equity',
  'post_ipo_equity',
  'post_ipo_debt',
  'post_ipo_secondary',
  'secondary_market'],
 'other': ['corporate_round', 'debt_financing', 'grant', 'series_unknown', 'undisclosed'],
#  'uncategorized': ['series_unknown', 'undisclosed']}
}
investment_type_to_stage = remap_dict(investment_stages)

In [34]:
investments_quarterly_df = pd.read_csv(OUTPUT_DIR / "cb_all_aggregated_funding_types_quarterly_startup.csv")
investments_df = pd.read_csv(OUTPUT_DIR / "cb_all_aggregated_funding_types_startup.csv")

all_funding_rounds_df = (
    pd.read_csv(OUTPUT_DIR / "cb_all_funding_rounds.csv")
    .assign(investment_stage = lambda df: df.investment_type.map(investment_type_to_stage))
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
)

In [7]:
sheet_id = "1m9_tKyJDaSy2vDWxYVP_9HlfBbGysQUV-xrb1FW3vok"
sheet_name = "crunchbase_check_v2"
cb_checks_df = google.access_google_sheet(sheet_id, sheet_name)

In [59]:
# (
#     investments_quarterly_df
#     .query("theme == 'Heat pumps'")
#     .pivot(index="quarter", columns="investment_type",values='raised_amount_gbp')
# )

In [60]:
investment_stages = ["early_stage", "growth_stage"]

exclude_ids = cb_checks_df.query("theme == 'Heat pumps'").query("`reviewer (karlis)` == 'no'").id.to_list()

(
    all_funding_rounds_df
    .query("theme == 'Heat pumps'")
    .query("investment_stage in @investment_stages")
    .query("org_id not in @exclude_ids")
    .drop_duplicates(subset=["funding_round_id"])
    .groupby(["year", "investment_stage"])
    .agg(
        raised_amount_gbp = ("raised_amount_gbp", "sum"),
    )
    .reset_index()
    .query("year >= 2014")
    .query("year <= 2025")
    .pivot(index="year", columns="investment_stage",values='raised_amount_gbp')
)

investment_stage,early_stage,growth_stage
year,,
2014,207.137767,31317.696140
2015,3638.109340,NaN
2016,1403.727294,23327.320506
2017,2862.656044,NaN
2018,5996.757328,NaN
2019,30466.883116,0.000000
2020,13872.953780,10858.800000
2021,48869.511269,212755.000000
2022,106050.116458,NaN


In [69]:
investment_stages = ["early_stage", "growth_stage"]

exclude_ids = cb_checks_df.query("theme == 'Low-Carbon Heating'").query("`reviewer (karlis)` == 'no'").id.to_list()

(
    all_funding_rounds_df
    .query("theme == 'Low-Carbon Heating'")
    .query("investment_stage in @investment_stages")
    .query("org_id not in @exclude_ids")
    .drop_duplicates(subset=["funding_round_id"])
    .groupby(["year"])
    .agg(
        raised_amount_gbp = ("raised_amount_gbp", "sum"),
    )
    .reset_index()
    .query("year >= 2014")
    .query("year <= 2025")
)

,year,raised_amount_gbp
10,2014,37451.978318
11,2015,22675.527267
12,2016,36249.799551
13,2017,41525.947080
14,2018,59327.489328
15,2019,84500.429747
16,2020,48708.257475
17,2021,317942.227876
18,2022,348644.092898
19,2023,513808.611789


In [62]:
# (
#     investments_df
#     .query("theme == 'Heat pumps'")
#     .pivot(index="year", columns="investment_type",values='raised_amount_gbp')
# )

In [63]:
themes = all_funding_rounds_df.theme.unique()
all_funding_rounds_filtered_df = []
for theme in themes:
    exclude_ids = cb_checks_df.query("theme == @theme").query("`reviewer (karlis)` == 'no'").id.to_list()
    all_funding_rounds_filtered_df.append(
        all_funding_rounds_df
        .query("theme == @theme")
        .query("org_id not in @exclude_ids")
        .drop_duplicates(subset=["funding_round_id"])
    )
all_funding_rounds_filtered_df = pd.concat(all_funding_rounds_filtered_df, ignore_index=True)

In [64]:
len(all_funding_rounds_df), len(all_funding_rounds_filtered_df)

(25740, 25681)

In [65]:
investment_stages = ["early_stage", "growth_stage"]

exclude_ids = cb_checks_df.query("theme == 'Heat pumps'").query("`reviewer (karlis)` == 'no'").id.to_list()

df = (
    all_funding_rounds_filtered_df
    .query("investment_stage in @investment_stages")
    .query("year >= 2020 and year <= 2024")
    .groupby(["theme", "investment_stage"])
    .agg(
        raised_amount_gbp = ("raised_amount_gbp", "sum"),
        counts = ("funding_round_id", "count"),
    )
    .reset_index()

)
df.pivot(
    index="theme",
    columns="investment_stage",
    values="raised_amount_gbp",
)

investment_stage,early_stage,growth_stage
theme,,
Biomass heating,0.000000e+00,NaN
CCUS,2.469726e+05,5.489520e+05
District heating,2.317906e+04,NaN
Energy storage,1.275161e+07,8.221680e+06
Geothermal energy,2.539477e+05,4.497714e+05
Heat pumps,5.958619e+05,4.428173e+05
Heat storage,3.077561e+05,NaN
Hydrogen energy,4.167588e+06,1.340241e+06
Hydrogen heating,1.468778e+03,NaN


In [66]:
df.pivot(
    index="theme",
    columns="investment_stage",
    values="counts",
)

investment_stage,early_stage,growth_stage
theme,,
Biomass heating,1.0,NaN
CCUS,83.0,5.0
District heating,15.0,NaN
Energy storage,1024.0,80.0
Geothermal energy,38.0,5.0
Heat pumps,90.0,6.0
Heat storage,46.0,NaN
Hydrogen energy,300.0,15.0
Hydrogen heating,2.0,NaN


In [55]:
# investment_types = ['early_stage', 'growth_stage']
# (
#     investments_df
#     .query("investment_type in @investment_types")
#     .query("year >= 2020 and year <= 2024")
#     .groupby(["theme", "investment_type"])
#     .agg(
#         {
#             "raised_amount_gbp": "sum",
#             "counts": "sum",
#         }
#     )
#     .reset_index()
#     .pivot(
#         index="theme",
#         columns="investment_type",
#         values="raised_amount_gbp",
#     )
# )

In [56]:
# investment_types = ['early_stage', 'growth_stage']
# (
#     investments_df
#     .query("investment_type in @investment_types")
#     .query("year >= 2020 and year <= 2024")
#     .groupby(["theme", "investment_type"])
#     .agg(
#         {
#             "raised_amount_gbp": "sum",
#             "counts": "sum",
#         }
#     )
#     .reset_index()
#     .pivot(
#         index="theme",
#         columns="investment_type",
#         values="counts",
#     )
# )

In [42]:
df = pd.read_csv(OUTPUT_DIR / "cb_growth_magnitude.csv")[["theme", "variable", "magnitude", "growth"]]
(
    df
    .query("variable == 'raised_amount_gbp_total'")
    .sort_values("magnitude", ascending=False)
    .head(10)
)

,theme,variable,magnitude,growth
46,Renewables,raised_amount_gbp_total,5674.913008,352.051589
42,Energy storage,raised_amount_gbp_total,4194.658418,448.823363
50,Solar,raised_amount_gbp_total,1697.978458,430.934937
26,Hydrogen energy,raised_amount_gbp_total,1101.565788,408.297908
58,Low-Carbon Heating,raised_amount_gbp_total,400.827537,742.815933
18,Heat pumps,raised_amount_gbp_total,210.221239,1136.817981
6,CCUS,raised_amount_gbp_total,159.184929,681.323834
14,Geothermal energy,raised_amount_gbp_total,140.743810,1031.321850
22,Heat storage,raised_amount_gbp_total,61.563722,179.649790
54,Wind,raised_amount_gbp_total,52.999956,306.679897


## Gateway to Research

In [79]:
gtr_ts_df = pd.read_csv(OUTPUT_DIR / "gtr_all_ts_df.csv")
gtr_ts_quarterly_df = pd.read_csv(OUTPUT_DIR / "gtr_all_ts_quarterly_df.csv")
categories_to_show = [
    "Low-carbon heating",
    "Wind",
    "Solar",
    "Hydrogen energy",
    "Energy efficiency",
    "CCUS",
]

# categories_to_show = [
#     "Heat pumps",
#     "District heating",
#     "Geothermal energy",
#     "Hydrogen heating",
#     "Solar thermal",
#     "Biomass heating",
#     "Solar thermal",
# ]

### Checked data

In [95]:
theme_to_title = {
    "wind": "Wind",
    "solar": "Solar",
    "hydrogen_energy": "Hydrogen energy",
    "energy_efficiency": "Energy efficiency",
    "ccus": "CCUS",
}
title_to_theme = {v: k for k, v in theme_to_title.items()}

In [108]:
checked_gtr_df_1 = google.access_google_sheet(sheet_id, "ukri_check")
checked_gtr_df_2 = google.access_google_sheet(sheet_id, "ukri_check_v2")
checked_gtr_df_lch = google.access_google_sheet(sheet_id, "ukri_check_v2_LCH")

In [110]:
gtr_data_df_1 = (
    checked_gtr_df_1
    # replace empty strings with NaN
    .replace("", pd.NA)
    .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
    # filter null
    .query("reviewer.notnull()")
)

gtr_data_df_2 = (
    checked_gtr_df_2
    # replace empty strings with NaN
    .replace("", pd.NA)
    .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
    .query("reviewer.notnull()")
)

gtr_data_checked_df = pd.concat([gtr_data_df_1, gtr_data_df_2], ignore_index=True)

_checked_gtr_df_lch = (
    checked_gtr_df_lch
    # replace empty strings with NaN
    .replace("", pd.NA)
    .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
    # filter null
    # .query("reviewer.notnull()")
)

In [99]:
gtr_all_projects_df = pd.read_csv(OUTPUT_DIR / "gtr_all_projects_df.csv")

In [135]:
gtr_all_projects_filtered_df = []
for category in categories_to_show[1:]:
    category_name = title_to_theme[category]
    exclude_ids = gtr_data_checked_df.query("theme == @category_name").query("reviewer == 'no'").id.to_list()
    print(f"{category_name} {len(exclude_ids)}") 
    df = (
        gtr_all_projects_df
        .query("theme == @category")
        .query("id not in @exclude_ids")
        .drop_duplicates(subset=["id"])
    )
    gtr_all_projects_filtered_df.append(df)

exclude_urls = _checked_gtr_df_lch[['url', 'reviewer']].query("reviewer == 'no'").url.to_list()
print(f"'Low-carbon heating' {len(exclude_urls)}") 
df_lch = (
    gtr_all_projects_df
    .query("theme == 'Low-carbon heating'")
    .query("url not in @exclude_urls")
    .drop_duplicates(subset=["id"])
)
gtr_all_projects_filtered_df.append(df_lch)
gtr_all_projects_filtered_df = (
    pd.concat(gtr_all_projects_filtered_df, ignore_index=True)
    .assign(year = lambda df: df.start.apply(lambda x: x[0:4]))
)

wind 7
solar 8
hydrogen_energy 14
energy_efficiency 10
ccus 1
'Low-carbon heating' 118


In [142]:
df = (
    gtr_all_projects_filtered_df
    .groupby(["theme", "year"])
    .agg(
        amount = ("amount", "sum"),
        counts = ("id", "count"),
    )
    .reset_index()
    .query("year >= '2018'")
    .query("year < '2025'")
    .assign(amount = lambda df: df.amount / 1e6)
)
df.pivot(index="year", columns="theme", values='amount')

theme,CCUS,Energy efficiency,Hydrogen energy,Low-carbon heating,Solar,Wind
year,,,,,,
2018,7.079839,56.483164,8.194674,8.400157,24.848347,23.227269
2019,30.466455,15.998363,62.377729,51.777261,43.150579,32.024585
2020,8.629245,35.872077,74.475276,37.060981,53.570984,24.791525
2021,107.944323,33.132918,165.164765,30.958047,33.285592,30.339314
2022,26.950102,8.755523,112.963692,16.576501,18.554553,32.478475
2023,34.583004,35.561176,176.376425,26.279736,40.661304,34.932825
2024,34.327423,56.449400,109.689990,39.316652,34.239612,49.310182


In [143]:
df.pivot(index="year", columns="theme", values='counts')

theme,CCUS,Energy efficiency,Hydrogen energy,Low-carbon heating,Solar,Wind
year,,,,,,
2018,28,48,39,41,91,53
2019,35,39,51,43,88,65
2020,39,98,86,61,105,114
2021,41,64,117,52,83,100
2022,58,67,160,62,90,116
2023,76,102,191,67,111,121
2024,65,72,116,61,98,89


In [80]:
gtr_ts_df

,time_period,year,n_projects,amount,amount_median,theme
0,2014-01-01,2014,3,0.823650,19935.0,Biomass heating
1,2015-01-01,2015,4,0.920713,123079.5,Biomass heating
2,2016-01-01,2016,0,0.000000,0.0,Biomass heating
3,2017-01-01,2017,0,0.000000,0.0,Biomass heating
4,2018-01-01,2018,0,0.000000,0.0,Biomass heating
...,...,...,...,...,...,...
175,2021-01-01,2021,67,39.046092,137852.0,Low-carbon heating
176,2022-01-01,2022,81,28.950314,50000.0,Low-carbon heating
177,2023-01-01,2023,75,27.462768,108833.0,Low-carbon heating
178,2024-01-01,2024,72,52.615544,260676.0,Low-carbon heating


In [152]:
urls_to_exclude = _checked_gtr_df_lch[['url', 'reviewer', 'title']].query("reviewer == 'no'").url.to_list()

(
    gtr_all_projects_df
    .query("theme == 'Heat pumps'")
    .query("url not in @urls_to_exclude")
    .drop_duplicates(subset=["id"])
    .assign(year = lambda df: df.start.apply(lambda x: x[0:4]))
    .groupby(["year"])
    .agg(
        amount = ("amount", "sum"),
        counts = ("id", "count"),
    )
    .reset_index()
)

,year,amount,counts
0,2014,262494.0,5
1,2015,4724986.0,12
2,2016,4755113.0,16
3,2017,2036834.0,5
4,2018,2470111.0,7
5,2019,17709996.0,12
6,2020,24708809.0,21
7,2021,12879630.0,22
8,2022,5278924.0,26
9,2023,12255419.0,23


In [72]:
chart_df = (
    gtr_ts_df
    .query("theme in @categories_to_show")
)
def normalise_ts(ts_df, base_year, variable):
    """normalise the data with respect to base_year"""
    normalised_ts_df = []
    for theme, theme_df in ts_df.groupby("theme"):
        theme_df = theme_df.copy()
        theme_df[variable] = theme_df[variable] / theme_df.query("year == @base_year")[variable].values[0]
        normalised_ts_df.append(theme_df)
    return pd.concat(normalised_ts_df, ignore_index=True)

# chart_df = normalise_ts(chart_df, 2014)

fig = charts.ts_smooth(
    ts = chart_df,
    variable="amount",
    variable_title="",
    category_column="theme",
    categories_to_show=categories_to_show,
    time_column="year",
)
charts.configure_plots(fig)

alt.Chart(...)

In [73]:
(
    chart_df
    .query("theme in @categories_to_show")
    .pivot(
        index="year",
        columns="theme",
        values="amount",
    )    
)

theme,Biomass heating,District heating,Geothermal energy,Heat pumps,Hydrogen heating,Solar thermal
year,,,,,,
2014,0.823650,4.830612,4.193432,0.262494,1.132308,1.611182
2015,0.920713,4.338242,0.331910,4.724986,0.234788,0.179838
2016,0.000000,2.403923,1.958961,4.755113,0.060481,7.420131
2017,0.000000,2.183818,5.308307,2.829541,4.545819,1.493807
2018,0.000000,1.726461,3.997729,2.470111,3.107294,0.488670
2019,0.000000,26.565069,4.181729,17.709996,13.748417,2.332866
2020,0.080810,9.865153,7.016635,24.923994,6.506993,2.020861
2021,0.248055,13.731691,4.505296,12.879630,17.557368,0.000000
2022,0.000000,2.865100,17.484619,5.456268,3.627847,0.070845


In [74]:
fig = charts.ts_smooth(
    ts = chart_df,
    variable="n_projects",
    variable_title="",
    category_column="theme",
    categories_to_show=categories_to_show,
    time_column="year",
)
charts.configure_plots(fig)

alt.Chart(...)

In [75]:
fig = charts.ts_smooth(
    ts = chart_df,
    variable="amount",
    variable_title="",
    category_column="theme",
    categories_to_show=["Heat pumps", "Low-carbon heating"],
    time_column="year",
)
charts.configure_plots(fig)

alt.Chart(...)

In [76]:
chart_df

,time_period,year,n_projects,amount,amount_median,theme
0,2014-01-01,2014,3,0.823650,19935.0,Biomass heating
1,2015-01-01,2015,4,0.920713,123079.5,Biomass heating
2,2016-01-01,2016,0,0.000000,0.0,Biomass heating
3,2017-01-01,2017,0,0.000000,0.0,Biomass heating
4,2018-01-01,2018,0,0.000000,0.0,Biomass heating
...,...,...,...,...,...,...
127,2021-01-01,2021,0,0.000000,0.0,Solar thermal
128,2022-01-01,2022,2,0.070845,35422.5,Solar thermal
129,2023-01-01,2023,4,0.991075,235515.5,Solar thermal
130,2024-01-01,2024,7,2.383744,234891.0,Solar thermal


In [77]:
(
    chart_df
    .query("theme in @categories_to_show")
    .pivot(
        index="year",
        columns="theme",
        values="n_projects",
    )    
)

theme,Biomass heating,District heating,Geothermal energy,Heat pumps,Hydrogen heating,Solar thermal
year,,,,,,
2014,3,4,10,5,3,5
2015,4,9,4,12,1,3
2016,0,9,4,16,1,13
2017,0,11,11,6,6,10
2018,0,6,10,7,8,9
2019,0,10,16,11,11,8
2020,1,15,15,23,20,8
2021,1,17,12,22,27,0
2022,0,12,18,29,22,2


In [78]:
chart_df.pivot(index="year", columns="theme",values='n_projects')

theme,Biomass heating,District heating,Geothermal energy,Heat pumps,Hydrogen heating,Solar thermal
year,,,,,,
2014,3,4,10,5,3,5
2015,4,9,4,12,1,3
2016,0,9,4,16,1,13
2017,0,11,11,6,6,10
2018,0,6,10,7,8,9
2019,0,10,16,11,11,8
2020,1,15,15,23,20,8
2021,1,17,12,22,27,0
2022,0,12,18,29,22,2


In [15]:
categories = ['Low-carbon heating']
chart_df = (
    gtr_ts_quarterly_df
    .query("theme in @categories")
)

fig = charts.ts_bar(
    ts = chart_df,
    variable="amount",
    variable_title="",
    time_column="quarter",
)
charts.configure_plots(fig)

alt.Chart(...)

In [16]:
categories = ['Low-carbon heating']
chart_df = (
    gtr_ts_quarterly_df
    .query("theme in @categories")
)

fig = charts.ts_bar(
    ts = chart_df,
    variable="n_projects",
    variable_title="",
    time_column="quarter",
)
charts.configure_plots(fig)

alt.Chart(...)

In [85]:
df = pd.read_csv(OUTPUT_DIR / "gtr_all_growth_magnitude_df.csv")[["theme", "variable", "magnitude", "growth"]]
df.query("variable == 'amount'")

,theme,variable,magnitude,growth
1,Biomass heating,amount,0.297698,1334.999381
4,CCUS,amount,42.486819,107.600238
7,District heating,amount,9.930073,-31.719636
10,Energy efficiency,amount,34.446271,-4.901021
13,Geothermal energy,amount,7.628876,75.192729
16,Heat pumps,amount,13.621289,-32.815821
19,Heat storage,amount,8.177963,-36.385026
22,Hydrogen energy,amount,127.734030,173.411784
25,Hydrogen heating,amount,8.916789,-12.169477
28,Micro CHP,amount,0.182791,-100.000000


In [86]:
df.query("variable == 'n_projects'")

,theme,variable,magnitude,growth
0,Biomass heating,n_projects,0.8,100.000000
3,CCUS,n_projects,54.4,104.210526
6,District heating,n_projects,13.8,19.354839
9,Energy efficiency,n_projects,79.2,40.229885
12,Geothermal energy,n_projects,16.6,36.585366
15,Heat pumps,n_projects,23.8,80.487805
18,Heat storage,n_projects,14.6,15.000000
21,Hydrogen energy,n_projects,131.8,164.161850
24,Hydrogen heating,n_projects,19.6,30.769231
27,Micro CHP,n_projects,0.2,-100.000000


## Policy trends

In [124]:
speeches_yearly_df = pd.read_csv(OUTPUT_DIR / "hansard_all_ts_df.csv")
speeches_quarterly_df = pd.read_csv(OUTPUT_DIR / "hansard_all_ts_quarterly_df.csv")
speeches_df = pd.read_csv(OUTPUT_DIR / "hansard_all_speeches_df.csv")

categories_to_show = [
    "Low-carbon heating",
    "Wind",
    "Solar",
    "Hydrogen energy",
    "Energy efficiency",
    "CCUS"
]

In [125]:
chart_df = (
    speeches_yearly_df
    .query("theme in @categories_to_show")
)

# chart_df = normalise_ts(chart_df, 2014, "speeches")

fig = charts.ts_smooth(
    ts = chart_df,
    variable="speeches",
    variable_title="",
    category_column="theme",
    categories_to_show=categories_to_show,
    time_column="year",
)
charts.configure_plots(fig)

alt.Chart(...)

In [126]:
chart_df.pivot(index="year", columns="theme",values='speeches')

theme,CCUS,Energy efficiency,Hydrogen energy,Low-carbon heating,Solar,Wind
year,,,,,,
2014,31,142,1,26,37,89
2015,81,111,6,17,47,203
2016,103,121,5,25,46,140
2017,38,205,11,13,17,20
2018,28,174,7,33,18,26
2019,74,143,22,39,56,46
2020,87,128,83,53,22,68
2021,109,179,106,110,26,77
2022,118,295,152,89,74,103


In [127]:
chart_df = (
    speeches_quarterly_df
    .query("theme in @categories_to_show")
)

fig = charts.ts_bar(
    ts = chart_df,
    variable="speeches",
    variable_title="",
    category_column="theme",
    categories_to_show=categories_to_show,
    time_column="quarter",
)
charts.configure_plots(fig)

alt.Chart(...)

In [128]:
category = 'Heat pumps'
chart_df = (
    speeches_quarterly_df
    .query("theme == @category")
)

fig = charts.ts_bar(
    ts = chart_df,
    variable="speeches",
    variable_title="",
    time_column="quarter",
)
charts.configure_plots(fig)

alt.Chart(...)

In [129]:
chart_df

,quarter,speeches,theme
81,2023-Q1,13,Heat pumps
82,2023-Q2,22,Heat pumps
83,2023-Q3,15,Heat pumps
84,2023-Q4,11,Heat pumps
85,2024-Q1,8,Heat pumps
86,2024-Q2,5,Heat pumps
87,2024-Q3,7,Heat pumps
88,2024-Q4,18,Heat pumps
89,2025-Q1,24,Heat pumps


In [130]:
category = 'Solar'
chart_df = (
    speeches_quarterly_df
    .query("theme == @category")
)

fig = charts.ts_bar(
    ts = chart_df,
    variable="speeches",
    variable_title="",
    time_column="quarter",
)
charts.configure_plots(fig, chart_title="Major increase in solar debates in Q1", chart_subtitle="Number of speeches in the House of Commons mentioning solar")

alt.Chart(...)

In [131]:
chart_df

,quarter,speeches,theme
144,2023-Q1,38,Solar
145,2023-Q2,12,Solar
146,2023-Q3,24,Solar
147,2023-Q4,10,Solar
148,2024-Q1,7,Solar
149,2024-Q2,16,Solar
150,2024-Q3,24,Solar
151,2024-Q4,40,Solar
152,2025-Q1,104,Solar


In [134]:
quarters =[
    '2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4',
    '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4',
    '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4',
    '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', 
    '2025-Q1'
]
category = "Heat pumps"

check_speeches_df = (
    speeches_df
    .query("theme == @category")
    .query("quarter in @quarters")
    .assign(speech_text_norm = lambda df: df.speech.apply(lambda x: re.sub(r"\s+", " ", x)))
    .drop_duplicates(["speakername", "date", "speech_text_norm"])
    .sort_values(["speech_id"])
)
check_speeches_df.to_csv(OUTPUT_DIR / "mission_radar_hansard_heat_pumps_2024-2025.csv", index=False)

In [137]:
check_speeches_df.groupby("quarter").agg(counts=("id", "count")).reset_index()

,quarter,counts
0,2021-Q1,4
1,2021-Q2,7
2,2021-Q3,7
3,2021-Q4,39
4,2022-Q1,16
5,2022-Q2,11
6,2022-Q3,4
7,2022-Q4,5
8,2023-Q1,13
9,2023-Q2,22


In [ ]:
"uk.org.publicwhip/debate/2024-11-04e.6.4"

In [61]:
check_speeches_df.columns

Index(['speech_id', 'speakername', 'speaker_id', 'person_id', 'speech', 'date',
       'year', 'major_heading', 'minor_heading', 'id', 'mission_labels',
       'topic_labels', 'quarter', 'theme'],
      dtype='object')

In [198]:
(
    check_speeches_df
    .groupby("major_heading")
    .agg(counts = ("speech_id", "count"))
    .sort_values("counts", ascending=False)

)

,counts
major_heading,
Energy Security and Net Zero,7
Gas Storage Levels,3
Cleat Hill Heat Pump Incident,3
New Homes (Solar Generation) Bill,3
Institute for Apprenticeships and Technical Education (Transfer of Functions etc) Bill [Lords],3
Geothermal Energy,3
COP29,2
Spray Foam Insulation: Property Value,2
"Housing, Communities and Local Government",2


In [65]:
speeches_df

,speech_id,speakername,speaker_id,person_id,speech,date,year,major_heading,minor_heading,id,mission_labels,topic_labels,quarter,theme
0,uk.org.publicwhip/debate/2015-06-25a.1018.6,Andrea Leadsom,NaN,uk.org.publicwhip/person/24829,I really do fail to understand why Opposition ...,2015-06-25,2015,ENERGY AND CLIMATE CHANGE,Renewable Energy,uk.org.publicwhip/debate/2015-06-25a.1018.6,ASF,Bioenergy,2015-Q2,Bioenergy
1,uk.org.publicwhip/debate/2018-05-01b.149.5,Claire Perry,NaN,uk.org.publicwhip/person/24915,My hon. Friend will be pleased to know that th...,2018-05-01,2018,"BUSINESS, ENERGY AND INDUSTRIAL STRATEGY",Topical Questions,uk.org.publicwhip/debate/2018-05-01b.149.5,ASF,Bioenergy,2018-Q2,Bioenergy
2,uk.org.publicwhip/debate/2018-10-16a.498.6,Claire Perry,NaN,uk.org.publicwhip/person/24915,My hon. Friend has made a valuable point. We h...,2018-10-16,2018,"BUSINESS, ENERGY AND INDUSTRIAL STRATEGY",Topical Questions,uk.org.publicwhip/debate/2018-10-16a.498.6,ASF,Bioenergy,2018-Q4,Bioenergy
3,uk.org.publicwhip/debate/2019-06-12e.660.7,Greg Clark,NaN,uk.org.publicwhip/person/11884,"I am very grateful, Mr Speaker, for your permi...",2019-06-12,2019,NET ZERO EMISSIONS TARGET,NaN,uk.org.publicwhip/debate/2019-06-12e.660.7,ASF,Training,2019-Q2,Bioenergy
4,uk.org.publicwhip/debate/2019-06-12e.679.4,Melanie Onn,NaN,uk.org.publicwhip/person/25317,What are the Government doing to support bioen...,2019-06-12,2019,NET ZERO EMISSIONS TARGET,NaN,uk.org.publicwhip/debate/2019-06-12e.679.4,ASF,Bioenergy,2019-Q2,Bioenergy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8450,uk.org.publicwhip/debate/2025-03-06b.458.1,Sarah Jones,NaN,uk.org.publicwhip/person/25673,"With permission, I would like to make a statem...",2025-03-06,2025,North Sea Energy,NaN,uk.org.publicwhip/debate/2025-03-06b.458.1,ASF,Training,2025-Q1,Wind
8451,uk.org.publicwhip/debate/2025-03-06b.464.1,Chi Onwurah,NaN,uk.org.publicwhip/person/24807,I congratulate the Minister on setting out a p...,2025-03-06,2025,North Sea Energy,NaN,uk.org.publicwhip/debate/2025-03-06b.464.1,ASF,Wind,2025-Q1,Wind
8452,uk.org.publicwhip/debate/2025-03-06b.464.4,Edward Leigh,NaN,uk.org.publicwhip/person/10352,When I climb the hill behind my home in the Li...,2025-03-06,2025,North Sea Energy,NaN,uk.org.publicwhip/debate/2025-03-06b.464.4,ASF,Wind,2025-Q1,Wind
8453,uk.org.publicwhip/debate/2025-03-06b.465.0,Sarah Jones,NaN,uk.org.publicwhip/person/25673,My hon. Friend raises a number of thorny issue...,2025-03-06,2025,North Sea Energy,NaN,uk.org.publicwhip/debate/2025-03-06b.465.0,ASF,Wind,2025-Q1,Wind


## Accuracy estimates

In [4]:
sheet_id = "1m9_tKyJDaSy2vDWxYVP_9HlfBbGysQUV-xrb1FW3vok"

In [10]:
PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}"

In [5]:
checked_gtr_df_1 = google.access_google_sheet(sheet_id, "ukri_check")
checked_gtr_df_2 = google.access_google_sheet(sheet_id, "ukri_check_v2")

In [6]:
gtr_data_df_1 = (
    checked_gtr_df_1
    # replace empty strings with NaN
    .replace("", pd.NA)
    .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
    # filter null
    .query("reviewer.notnull()")
)

In [23]:
themes = gtr_data_df_1.theme.unique()
dfs = []
for theme in themes:
    _gtr_data_df = gtr_data_df_1.query("theme == @theme")
    df = pd.read_json(OUTPUT_DIR / f"gtr_llm_check_v2_{theme}.jsonl", lines=True)
    _df = _gtr_data_df.merge(df[["id", "is_relevant"]], on="id", how="left", suffixes=("_",""))
    dfs.append(_df)
dfs = pd.concat(dfs, ignore_index=True)[["id", "theme", "reviewer", "is_relevant"]]

In [24]:
gtr_data_df_2 = (
    checked_gtr_df_2
    # replace empty strings with NaN
    .replace("", pd.NA)
    .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
    .query("reviewer.notnull()")
)[["id", "theme", "reviewer", "is_relevant"]]

gtr_data_checked_df = pd.concat([dfs, gtr_data_df_2], ignore_index=True).drop_duplicates(subset=["id"])

In [25]:
gtr_data_checked_df.value_counts("theme")

theme
hydrogen_energy      40
ccus                 20
district_heating     20
energy_efficiency    20
energy_storage       20
geothermal_energy    20
hydrogen_heating     20
solar                20
solar_thermal        20
wind                 20
heat_pumps           19
micro_chp            15
biomass_heating      11
Name: count, dtype: int64

In [163]:
config_name = [
    "ccus",
    "energy_efficiency",
    "energy_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "solar",
    "solar_thermal",
    "wind",
    "micro_chp",
]

In [36]:
# replace yes with True and no with False
gtr_data_checked_df = gtr_data_checked_df.replace({"yes": True, "no": False})

/var/folders/5s/x3974l2x1hz1t5fs87qyxjgw0000gp/T/ipykernel_44127/1947288510.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gtr_data_checked_df = gtr_data_checked_df.replace({"yes": True, "no": False})


In [38]:
# import precision and recall metrics
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import classification_report

print(precision_score(y_true=gtr_data_checked_df.reviewer, y_pred=gtr_data_checked_df.is_relevant))
print(recall_score(y_true=gtr_data_checked_df.reviewer, y_pred=gtr_data_checked_df.is_relevant))
print(classification_report(y_true=gtr_data_checked_df.reviewer, y_pred=gtr_data_checked_df.is_relevant))

0.9364161849710982
0.8223350253807107
              precision    recall  f1-score   support

       False       0.62      0.84      0.71        68
        True       0.94      0.82      0.88       197

    accuracy                           0.83       265
   macro avg       0.78      0.83      0.79       265
weighted avg       0.86      0.83      0.83       265



In [26]:
(gtr_data_checked_df.is_relevant == gtr_data_checked_df.reviewer).mean()

0.8264150943396227

In [27]:
(gtr_data_checked_df.query("is_relevant == 'yes'").is_relevant == gtr_data_checked_df.query("is_relevant == 'yes'").reviewer).mean()

0.9364161849710982

In [ ]:
(gtr_data_checked_df.query("is_relevant == 'yes'").is_relevant) == gtr_data_checked_df.query("is_relevant == 'yes'").reviewer).mean()

In [28]:
(gtr_data_checked_df.query("is_relevant == 'no'").is_relevant == gtr_data_checked_df.query("is_relevant == 'no'").reviewer).mean()

0.6195652173913043

In [29]:
for theme in gtr_data_checked_df.theme.unique():
    print(theme)
    df = gtr_data_checked_df.query("theme == @theme")
    accuracy = (df.is_relevant == df.reviewer).mean()
    print(f"accuracy: {accuracy:.2f}")

biomass_heating
accuracy: 0.73
district_heating
accuracy: 0.95
geothermal_energy
accuracy: 0.90
heat_pumps
accuracy: 0.89
hydrogen_energy
accuracy: 0.88
ccus
accuracy: 0.55
energy_efficiency
accuracy: 0.90
hydrogen_heating
accuracy: 0.90
micro_chp
accuracy: 0.93
solar_thermal
accuracy: 0.70
energy_storage
accuracy: 0.60
solar
accuracy: 0.90
wind
accuracy: 0.85


In [ ]:
gtr_data_df_2.query("is_relev")

# Extra charts

In [ ]:
magnitude_growth_df = pd.read_csv(OUTPUT_DIR / "cb_growth_magnitude.csv")
magnitude_growth_quarterly = pd.read_csv(OUTPUT_DIR / "cb_growth_magnitude_quarterly.csv")
magnitude_growth_quarterly.sort_values("magnitude", ascending=False)


In [ ]:
# Create a selection dropdown
variable_selection = alt.binding_select(options=magnitude_growth_df['variable'].unique().tolist(), name='Variable:')
variable_select = alt.selection_point(fields=['variable'], bind=variable_selection, name="variable_selection")

# Base chart
base = alt.Chart(magnitude_growth_quarterly).transform_filter(variable_select).encode(
    x=alt.X("magnitude", title="Magnitude"),
    y=alt.Y("growth", title="Growth"),
)

# Points and text
points = base.mark_circle()
text = base.mark_text(align="left", baseline="middle", dx=7).encode(
    text="theme",
)

# Combine points and text
fig = (points + text).add_params(variable_select).properties(width=800, height=400).interactive()

charts.configure_plots(fig, "Quarterly investment trends", "Growth compares 2025-Q1 and average 2024-Q1 to 2024-Q4")


In [ ]:
magnitude_growth_df = pd.read_csv(OUTPUT_DIR / "gtr_all_growth_magnitude_df.csv")
magnitude_growth_quarterly = pd.read_csv(OUTPUT_DIR / "gtr_all_growth_magnitude_quarterly_df.csv")
magnitude_growth_quarterly.query("variable=='n_projects'").sort_values("magnitude", ascending=False)

In [ ]:
magnitude_growth_quarterly.query("variable=='amount'").sort_values("magnitude", ascending=False)

In [ ]:
import altair as alt
fig = (
    alt.Chart(magnitude_growth_df.query("variable == 'amount'"))
    .mark_circle()
    .encode(
        x=alt.X("magnitude"),
        y=alt.Y("growth"),
    )
    # add labels
    .mark_text(align="left", baseline="middle", dx=7).encode(
        text="theme",
    )
    .properties(width=800, height=400)
)
fig.interactive()